# Multilingual news recommender: A to Z

The project name is historical; this is a standalone portfolio, not an enterprise product. Follow the data, tokens, adapter, generated facts, learned links, embeddings and recommendations in order.

<a id="setup"></a>
## 1. Locate the project

Run from the top with the project environment selected. Nothing here retrains automatically. Local notebooks need installed dependencies and downloaded artifacts; website visitors do not.

In [ ]:
from pathlib import Path
import sys, os, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "news_pipeline.py").exists())
sys.path.insert(0, str(ROOT))
os.environ.setdefault("NEWS_EMBEDDING_DEVICE", "cpu")
from src.portfolio_config import ARTIFACTS, BASE_MODEL, EMBEDDING_MODEL, configure_cache, read_json, read_jsonl
configure_cache()
from src.portfolio_reports import evidence_report
print("Project:", ROOT)
print("Artifacts:", ARTIFACTS)


<a id="evidence"></a>
## 2. Check what actually exists

Read measured artifact manifests, not a claim that every experiment succeeded. Missing training, review, or deployment remains visible. Historical mock metrics are not real-news results.

In [ ]:
report = evidence_report()
display(report)

<a id="inputs"></a>
## 3. One real article and your own interest

The interest sentence belongs to the user. Required filters are optional structured constraints, not Qwen output. This first view uses a recorded real inference example; its source and model identity remain attached.

In [ ]:
examples = read_json(ROOT / "data" / "release" / "example_analyses.json", [])
if not examples:
    raise RuntimeError("No genuine inference examples yet. Run scripts/build_news_release.py after setup.")
analysis = examples[0]
article = analysis["article"]
interest = "Iran nuclear diplomacy and international sanctions"
required_filters = {"countries": ["Iran"]}
display({"article": article, "user_interest": interest, "required_filters": required_filters})

<a id="tokens"></a>
## 4. Qwen: text becomes token IDs

This is the same text-to-token step as a text-generation LLM course. Qwen's tokenizer is unchanged. Arabic names may use several subword tokens; that is normal. Fine-tuning changes model behavior, not these token boundaries.

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=False)
text = article["title"] + "\n\n" + article["body"]
token_ids = tokenizer.encode(text, add_special_tokens=False)
display({"model": BASE_MODEL, "vocabulary_size": len(tokenizer), "article_tokens": len(token_ids)})
display(list(zip(tokenizer.convert_ids_to_tokens(token_ids[:30]), token_ids[:30])))

<a id="adapter"></a>
## 5. QLoRA: what was actually trained

Qwen is the pretrained generator. LoRA is the small set of trainable matrices attached to it; QLoRA keeps the frozen base in 4-bit during training. Article/JSON examples train the adapter through Qwen. Lower loss and changed weights prove optimization happened, not that quality is sufficient. Phase 3 selected extraction-only-v3 using the predeclared validation gate. This is a research candidate being applied in Phase 4, not a production claim.

In [ ]:
for run in report["extraction_experiments"]:
    display(run)
    loss_path = ARTIFACTS / "runs" / run["run_id"] / "loss.jsonl"
    loss = read_jsonl(loss_path) if loss_path.exists() else []
    display(loss[:3] + loss[-3:] if loss else "No saved optimizer steps")
from src.phase3_configuration import selected_configuration
selection = selected_configuration()
display({"validated_phase3_selection": selection["run_id"] if selection else None,
         "selected_adapter": str(selection["adapter_path"]) if selection else None,
         "displayed_example_model": analysis["model"]})

<a id="extraction"></a>
## 6. Qwen output: generated facts with source evidence

These JSON fields were generated in the recorded run, not hand-entered article tags. Cached inference is legitimate when tied to identical text and model versions. Unsupported predictions are shown and excluded from filters. Run the optional live cell at the end to test genuinely new text.

In [ ]:
display({"generated_at": analysis["generated_at"], "cache_key": analysis["cache_key"],
         "raw_generated_text": analysis["raw_model_outputs"], "parsed_json": analysis["extraction"],
         "grounded_passages": analysis["evidence"], "excluded_predictions": analysis["ungrounded_predictions"]})

<a id="linking"></a>
## 7. Learned entity linking

E5 retrieves candidate entity descriptions. A supervised logistic classifier scores context and name features and can abstain. Known aliases remain useful features, but are not a manually configured final lookup decision. The catalogue supplies IDs and is limited to entities in the training partition. An unresolved name is not a made-up entity.

In [ ]:
from src.phase3_configuration import selected_configuration
selection = selected_configuration()
display(analysis["entity_links"])
display(read_json(selection["linker_path"] / "metrics.json", {"status": "metrics unavailable"}))

<a id="filters"></a>
## 8. Qwen + linker: the structured recommendation contribution

Put the user's requested filters next to article tags. A direct match means every requested category has at least one match. These are application comparisons of model-derived facts, not an E5 prediction. Two names co-occurring do not prove a contract or other relationship.

In [ ]:
from src.news_pipeline import compare_filters
comparison, coverage, direct = compare_filters(required_filters, analysis["tags"])
display(comparison)
display({"required_categories_covered": coverage, "all_required_categories_match": direct})

<a id="e5"></a>
## 9. A different pretrained model: multilingual E5

E5 compares your written interest with the article's natural text, without adding the exact filters. It directly embeds Arabic and English in a shared space; there is no manual translation. It is not fine-tuned here. Long articles are chunked and their normalized vectors are averaged. The dot product is similarity, not a probability.

In [ ]:
from src.learned_linker import encode_batch
query_text = "query: " + interest
passage_text = "passage: " + text
vectors = encode_batch([query_text, passage_text])
display({"query": query_text, "passage": passage_text, "vector_shape": vectors.shape,
         "first_8_user_values": vectors[0, :8].tolist(), "first_8_article_values": vectors[1, :8].tolist(),
         "cosine_similarity": float(vectors[0] @ vectors[1])})

<a id="phrases"></a>
## 10. Show supporting phrase similarities

These pairs are calculated by E5 for the current texts, not manually authored translations. They are independent phrase probes, not a definitive explanation of attention or causality. They inspect a bounded set of phrases and can be wrong.

In [ ]:
from src.embeddings import find_e5_phrase_matches
display(find_e5_phrase_matches(interest, text))

<a id="ranking"></a>
## 11. Rank the current processed collection

The same NewsPipeline powers the live app. It compares a fresh user vector with stored article vectors. Hybrid ranking combines E5 with exact filter coverage; without filters it equals E5. Only records matching the current model/configuration enter the live index. If the index is stale, rebuild it rather than mixing old and new model results.

In [ ]:
from src.news_pipeline import NewsPipeline
pipeline = NewsPipeline()
ranking = pipeline.recommend(interest, required_filters, limit=5)
display(ranking)
second_interest = "Elections and constitutional reform"
display(pipeline.recommend(second_interest, limit=5))

<a id="live"></a>
## 12. Optional: analyze genuinely new article text

Set RUN_LIVE to True and replace the text. This performs fresh Qwen generation and linking; on a 1080 Ti it can take minutes. Your text is not persisted or used for training by default. Training holds a GPU lock, so wait for that job to finish. Failures remain failures, never prewritten tags.

In [ ]:
RUN_LIVE = False
new_article = {"title": "Replace with your article title", "body": "Replace this sentence with a real article you may use."}
if RUN_LIVE:
    live = pipeline.analyze_article(new_article, persist=False)
    display(live)
    display(pipeline.explain(interest, live, required_filters))
else:
    print("Live generation not requested. The earlier JSON was a labeled saved model run.")

<a id="evaluation"></a>
## 13. What counts as evidence

JSON parsing measures syntax; schema validity measures required fields and types; entity F1 measures extracted content against independently reviewed labels. The linker has its own accuracy/abstention metrics. Recommendations use Recall@5 and nDCG@5 on the three fixed judged pools. All 30 extraction and 30 recommendation references have human judgments; the pools are small and machine-assisted training labels remain separate.

In [ ]:
display(evidence_report())